# LLM Prover
> Your AI truth layer - benchmark, monitor, and prove how every model performs on YOUR data

*v0.3.0*

In [ ]:
# @title Setup
# @markdown **Run this cell every session.** First time? See instructions below.
# @markdown ---
# @markdown **First time setup:** [Getting started guide](https://llmprover.pysolvr.com/docs/getting-started) |
# @markdown Save a copy: File > Save a copy in Drive |
# @markdown Add API key: click key icon (left sidebar) > add `LLMPROVER_API_KEY`
# @markdown ---
!pip install -q pysolvr-client
import sys
from google.colab import userdata, drive

try:
    API_KEY = userdata.get('LLMPROVER_API_KEY')
except userdata.SecretNotFoundError:
    from IPython.display import HTML, display
    display(HTML('<div style="background:#1e293b;border:1px solid #ef4444;border-radius:8px;padding:16px;font-family:Inter,system-ui,sans-serif;color:#f1f5f9"><b style="color:#ef4444">API key not found in Colab Secrets</b><ol style="color:#94a3b8;font-size:13px;margin-top:8px"><li>Click the key icon in the left sidebar</li><li>Add secret: <code>LLMPROVER_API_KEY</code></li><li>Paste your API key as the value</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'API key not configured - see instructions above'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://llmprover.pysolvr.com/api'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#8B5CF6', '#06B6D4')
drive_mgr = DriveManager('llmprover', ['comparisons', 'benchmarks', 'reports', 'evaluations'])
drive_mgr.ensure_folders()

if client.health_check():
    ui.success('Connected to LLM Prover API', f'Drive: {drive_mgr.root}')
else:
    ui.error('Could not connect to API', 'Check your API key and try again')

In [ ]:
# @title Compare Models
# @markdown Test your prompt across multiple LLMs in parallel
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip():
    ui.error('Enter a prompt above and re-run this cell')
else:
    payload = {'prompt': PROMPT, 'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}}
    if CONTEXT.strip():
        payload['context'] = CONTEXT
    result = client.run_async('/compare', payload)
    if result['ok']:
        d = result['data']
        rows = []
        for r in d.get('results', []):
            preview = (r.get('result_text') or '')[:120]
            if r.get('error'):
                preview = f"ERROR: {r['error'][:80]}"
            rows.append({'Provider': r['provider'], 'Model': r.get('model',''), 'Latency': f"{r.get('latency_ms',0)}ms", 'Cost': f"${r.get('cost_usd',0):.4f}", 'Output': preview})
        results_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in ['Provider','Model','Latency','Cost','Output']) + '</tr></thead><tbody>'
        for row in rows:
            results_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in ['Provider','Model','Latency','Cost','Output']) + '</tr>'
        results_html += '</tbody></table>'
        enr = d.get('enrichments', {})
        enr_html = f"<p><b>Recommendation:</b> {enr.get('recommendation','N/A')}</p>"
        enr_html += f"<p><b>Fastest:</b> {enr.get('fastest','N/A')} | <b>Cheapest:</b> {enr.get('cheapest','N/A')}</p>"
        if enr.get('agreement_score') is not None:
            enr_html += f"<p><b>Agreement:</b> {enr['agreement_score']:.0%}</p>"
        full_html = ''
        for r in d.get('results', []):
            text = r.get('result_text') or r.get('error') or 'No output'
            full_html += f"<details><summary><b>{r['provider']}</b> ({r.get('model','')})</summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px;margin:8px 0'>{text}</pre></details>"
        tabs = {'Results': results_html, 'Analysis': enr_html, 'Full Outputs': full_html}
        ui.tabs(tabs)
        ui.cost_badge(sum(r.get('tokens_total',0) for r in d.get('results',[])), d.get('total_cost_usd',0))
        import json as _json
        cid = d['comparison_id']
        drive_mgr.save('comparisons', f'{cid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'total_cost_usd': d.get('total_cost_usd',0), 'providers': d.get('providers_responded',[])})
        ui.success(f'Saved to Drive: comparisons/{cid}.json')
    else:
        ui.error(result.get('error', 'Compare failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Verdictator
# @markdown Score and rank model responses against your expected output
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
EXPECTED_OUTPUT = ""  # @param {type:"string"}
PROVIDERS = "All"  # @param ["All", "Grok", "Anthropic", "OpenAI"]
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip() or not CONTEXT.strip() or not EXPECTED_OUTPUT.strip():
    ui.error('Fill in PROMPT, CONTEXT, and EXPECTED_OUTPUT above, then re-run')
else:
    payload = {
        'prompt': PROMPT,
        'context': CONTEXT,
        'mode': 'gold_standard',
        'expected_output': EXPECTED_OUTPUT,
        'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}
    }
    if PROVIDERS != 'All':
        payload['providers'] = [PROVIDERS.lower()]
    result = client.run_async('/evaluate', payload)
    if result['ok']:
        d = result['data']
        rows = []
        for r in d.get('results', []):
            rows.append({
                'Provider': r['provider'],
                'Model': r.get('model', ''),
                'Quality': f"{r.get('quality_score', 0):.1f}",
                'Value': f"{r.get('value_score', 0):.0f}",
                'Cost': f"${r.get('cost_usd', 0):.4f}",
                'Latency': f"{r.get('latency_ms', 0)}ms"
            })
        w = d.get('winner', {})
        winner_html = f"<p><b>Best quality:</b> {w.get('by_quality', 'N/A')} | <b>Best value:</b> {w.get('by_value', 'N/A')}</p>"
        if w.get('reasoning'):
            winner_html += f"<p style='color:#94a3b8;font-size:12px'>{w['reasoning']}</p>"
        breakdown_html = ''
        for r in d.get('results', []):
            bd = r.get('breakdown', {})
            if bd:
                breakdown_html += f"<details><summary><b>{r['provider']}</b> (quality: {r.get('quality_score',0):.1f})</summary>"
                breakdown_html += '<ul>' + ''.join(f"<li>{k}: {v:.1f}</li>" for k, v in bd.items()) + '</ul></details>'
        responses_html = ''
        for r in d.get('results', []):
            text = r.get('response_text', 'No output')
            responses_html += f"<details><summary><b>{r['provider']}</b></summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px'>{text}</pre></details>"
        cols = ['Provider', 'Model', 'Quality', 'Value', 'Cost', 'Latency']
        table_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in cols) + '</tr></thead><tbody>'
        for row in rows:
            table_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in cols) + '</tr>'
        table_html += '</tbody></table>'
        tabs = {'Scores': table_html + winner_html, 'Breakdown': breakdown_html or '<p>No breakdown available</p>', 'Responses': responses_html}
        judge_used = d.get('judge_used', False)
        if judge_used:
            tabs['Scores'] += '<p style="font-size:11px;color:#94a3b8">Judge scoring applied (Pro+)</p>'
        ui.tabs(tabs)
        ui.cost_badge(0, d.get('total_cost_usd', 0))
        import json as _json
        eid = d.get('evaluation_id', d.get('collection_hash', 'unknown'))
        drive_mgr.save('evaluations', f'{eid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'mode': 'gold_standard', 'collection_hash': d.get('collection_hash', ''), 'total_cost_usd': d.get('total_cost_usd', 0)})
        ui.success(f'Saved to Drive: evaluations/{eid}.json')
    else:
        ui.error(result.get('error', 'Evaluation failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Rubric Evaluation
# @markdown Score responses against your custom rubric criteria
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
RUBRIC = ""  # @param {type:"string"}
PROVIDERS = "All"  # @param ["All", "Grok", "Anthropic", "OpenAI"]
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip() or not CONTEXT.strip() or not RUBRIC.strip():
    ui.error('Fill in PROMPT, CONTEXT, and RUBRIC above, then re-run')
else:
    payload = {
        'prompt': PROMPT,
        'context': CONTEXT,
        'mode': 'rubric',
        'rubric': RUBRIC,
        'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}
    }
    if PROVIDERS != 'All':
        payload['providers'] = [PROVIDERS.lower()]
    result = client.run_async('/evaluate', payload)
    if result['ok']:
        d = result['data']
        rows = []
        for r in d.get('results', []):
            rows.append({
                'Provider': r['provider'],
                'Model': r.get('model', ''),
                'Quality': f"{r.get('quality_score', 0):.1f}",
                'Value': f"{r.get('value_score', 0):.0f}",
                'Cost': f"${r.get('cost_usd', 0):.4f}",
                'Latency': f"{r.get('latency_ms', 0)}ms"
            })
        w = d.get('winner', {})
        winner_html = f"<p><b>Best quality:</b> {w.get('by_quality', 'N/A')} | <b>Best value:</b> {w.get('by_value', 'N/A')}</p>"
        if w.get('reasoning'):
            winner_html += f"<p style='color:#94a3b8;font-size:12px'>{w['reasoning']}</p>"
        breakdown_html = ''
        for r in d.get('results', []):
            bd = r.get('breakdown', {})
            if bd:
                breakdown_html += f"<details><summary><b>{r['provider']}</b> (quality: {r.get('quality_score',0):.1f})</summary>"
                breakdown_html += '<ul>' + ''.join(f"<li>{k}: {v:.1f}</li>" for k, v in bd.items()) + '</ul></details>'
        responses_html = ''
        for r in d.get('results', []):
            text = r.get('response_text', 'No output')
            responses_html += f"<details><summary><b>{r['provider']}</b></summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px'>{text}</pre></details>"
        cols = ['Provider', 'Model', 'Quality', 'Value', 'Cost', 'Latency']
        table_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in cols) + '</tr></thead><tbody>'
        for row in rows:
            table_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in cols) + '</tr>'
        table_html += '</tbody></table>'
        tabs = {'Scores': table_html + winner_html, 'Breakdown': breakdown_html or '<p>No breakdown available</p>', 'Responses': responses_html}
        judge_used = d.get('judge_used', False)
        if judge_used:
            tabs['Scores'] += '<p style="font-size:11px;color:#94a3b8">Judge scoring applied (Pro+)</p>'
        ui.tabs(tabs)
        ui.cost_badge(0, d.get('total_cost_usd', 0))
        import json as _json
        eid = d.get('evaluation_id', d.get('collection_hash', 'unknown'))
        drive_mgr.save('evaluations', f'{eid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'mode': 'rubric', 'collection_hash': d.get('collection_hash', ''), 'total_cost_usd': d.get('total_cost_usd', 0)})
        ui.success(f'Saved to Drive: evaluations/{eid}.json')
    else:
        ui.error(result.get('error', 'Evaluation failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Compliance Evaluation (Enterprise)
# @markdown Evaluate model responses against regulatory/policy requirements
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
REGULATIONS = ""  # @param {type:"string"}
PROVIDERS = "All"  # @param ["All", "Grok", "Anthropic", "OpenAI"]
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip() or not CONTEXT.strip() or not REGULATIONS.strip():
    ui.error('Fill in PROMPT, CONTEXT, and REGULATIONS above, then re-run')
else:
    payload = {
        'prompt': PROMPT,
        'context': CONTEXT,
        'mode': 'compliance',
        'regulations': REGULATIONS,
        'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}
    }
    if PROVIDERS != 'All':
        payload['providers'] = [PROVIDERS.lower()]
    result = client.run_async('/evaluate', payload)
    if result['ok']:
        d = result['data']
        rows = []
        for r in d.get('results', []):
            rows.append({
                'Provider': r['provider'],
                'Model': r.get('model', ''),
                'Quality': f"{r.get('quality_score', 0):.1f}",
                'Value': f"{r.get('value_score', 0):.0f}",
                'Cost': f"${r.get('cost_usd', 0):.4f}",
                'Latency': f"{r.get('latency_ms', 0)}ms"
            })
        w = d.get('winner', {})
        winner_html = f"<p><b>Best quality:</b> {w.get('by_quality', 'N/A')} | <b>Best value:</b> {w.get('by_value', 'N/A')}</p>"
        if w.get('reasoning'):
            winner_html += f"<p style='color:#94a3b8;font-size:12px'>{w['reasoning']}</p>"
        breakdown_html = ''
        for r in d.get('results', []):
            bd = r.get('breakdown', {})
            if bd:
                breakdown_html += f"<details><summary><b>{r['provider']}</b> (quality: {r.get('quality_score',0):.1f})</summary>"
                breakdown_html += '<ul>' + ''.join(f"<li>{k}: {v:.1f}</li>" for k, v in bd.items()) + '</ul></details>'
        responses_html = ''
        for r in d.get('results', []):
            text = r.get('response_text', 'No output')
            responses_html += f"<details><summary><b>{r['provider']}</b></summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px'>{text}</pre></details>"
        cols = ['Provider', 'Model', 'Quality', 'Value', 'Cost', 'Latency']
        table_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in cols) + '</tr></thead><tbody>'
        for row in rows:
            table_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in cols) + '</tr>'
        table_html += '</tbody></table>'
        tabs = {'Scores': table_html + winner_html, 'Breakdown': breakdown_html or '<p>No breakdown available</p>', 'Responses': responses_html}
        judge_used = d.get('judge_used', False)
        if judge_used:
            tabs['Scores'] += '<p style="font-size:11px;color:#94a3b8">Judge scoring applied (Pro+)</p>'
        ui.tabs(tabs)
        ui.cost_badge(0, d.get('total_cost_usd', 0))
        import json as _json
        eid = d.get('evaluation_id', d.get('collection_hash', 'unknown'))
        drive_mgr.save('evaluations', f'{eid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'mode': 'compliance', 'collection_hash': d.get('collection_hash', ''), 'total_cost_usd': d.get('total_cost_usd', 0)})
        ui.success(f'Saved to Drive: evaluations/{eid}.json')
    else:
        ui.error(result.get('error', 'Evaluation failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Available Models
result = client.call('GET', '/models')
if result['ok']:
    rows = []
    for pname, pdata in result['data'].get('providers', {}).items():
        for m in pdata.get('models', []):
            if m.get('status') == 'eol':
                continue
            default = ' (default)' if m.get('id') == pdata.get('default') else ''
            rows.append({'Provider': pname, 'Model': m['id'] + default, 'Input $/1K': f"${m.get('cost_per_1k_input','?')}", 'Output $/1K': f"${m.get('cost_per_1k_output','?')}", 'Max Tokens': m.get('max_tokens','')})
    ui.table(rows, ['Provider', 'Model', 'Input $/1K', 'Output $/1K', 'Max Tokens'])
else:
    ui.error(result.get('error', 'Could not fetch models'))


In [ ]:
# @title Account
# @markdown Subscription, usage, and support
ACTION = 'Subscription'  # @param ["Subscription", "Usage", "Support"]

if ACTION == 'Subscription':
    result = client.call('GET', '/account')
    if result['ok']:
        d = result['data']
        ui.card('Subscription', f"<b>{d.get('plan', 'Free')}</b><br>"
            f"Status: {d.get('status', 'active')}<br>"
            f"<a href='https://llmprover.pysolvr.com/pricing' style='color:#60a5fa'>Manage plan</a>",
            status='success')
    else:
        ui.error(result.get('error', 'Could not fetch account'), actions=result.get('actions'))

elif ACTION == 'Usage':
    result = client.get_usage()
    if result['ok']:
        data = result['data']
        ui.usage_bar(data.get('monthly_spend_usd', 0), data.get('monthly_limit_usd', 1))
        if data.get('recent'):
            ui.table(data['recent'])
    else:
        ui.error(result.get('error', 'Could not fetch usage'), actions=result.get('actions'))

elif ACTION == 'Support':
    from IPython.display import HTML, display
    display(HTML('<div style="font-family:Inter,system-ui,sans-serif;color:#f1f5f9;font-size:13px">'
        '<p><b>Files:</b> Google Drive > pysolvr > llmprover</p>'
        '<p><b>Docs:</b> <a href="https://llmprover.pysolvr.com/docs" style="color:#60a5fa">https://llmprover.pysolvr.com/docs</a></p>'
        '<p><b>Support:</b> <a href="mailto:support@pysolvr.com?subject=LLM Prover" style="color:#60a5fa">support@pysolvr.com</a></p>'
        '</div>'))
